# Notebook 05 — SOH Target Calculation

**Objective:** Compute State-of-Health (SOH) for every discharge cycle.

**Formula:**
$$
\text{SOH}(\%) = \frac{\text{Current Capacity}}{\text{Initial Capacity}} \times 100
$$

**Initial capacity** is set to the **maximum observed capacity per battery**.
This is more robust than using the first-cycle capacity for the NASA dataset,
because several batteries (B0033, B0041 etc.) have anomalously low first-cycle
readings due to mixed discharge protocols.

**Leakage note:** `capacity` is used ONLY to compute the SOH target.
It is NOT used as a model feature in any downstream notebook.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from soh import add_soh, soh_stats, flag_non_monotonic, EOL_THRESHOLD

sns.set_style('whitegrid')
plt.rcParams.update({'font.size': 11})
print('Libraries loaded successfully')

## 1. Load the Featured Dataset

In [ ]:
feat_path = os.path.join(ROOT, 'cleaned_data', 'featured_dataset.csv')
feat = pd.read_csv(feat_path)
feat = feat.sort_values(['battery_id', 'cycle']).reset_index(drop=True)

print(f'Shape: {feat.shape}')
print(f'Batteries: {feat["battery_id"].nunique()}')
print(f'Columns: {list(feat.columns)}')
feat.head()

## 2. Compute SOH (Initial Capacity = Max per Battery)

In [ ]:
feat = add_soh(feat)   # adds: initial_capacity, soh

print('New columns added:', ['initial_capacity', 'soh'])
print(f'SOH range: {feat["soh"].min():.2f}% – {feat["soh"].max():.2f}%')
print(f'SOH mean:  {feat["soh"].mean():.2f}%  |  std: {feat["soh"].std():.2f}%')
feat[['battery_id', 'cycle', 'capacity', 'initial_capacity', 'soh']].head(10)

## 3. SOH Statistics per Battery

In [ ]:
stats = soh_stats(feat)
stats

## 4. Flag Non-Monotonic Batteries

Some batteries show a rising SOH pattern. This indicates mixed discharge
protocols (different cutoff voltages or discharge rates) rather than
genuine regeneration. These batteries are INCLUDED in training but documented.

In [ ]:
non_mono = flag_non_monotonic(feat)
print(f'Non-monotonic batteries ({len(non_mono)}):')
for b in sorted(non_mono):
    row = stats.loc[b]
    print(f'  {b}: SOH {row["soh_min"]:.1f}% – {row["soh_max"]:.1f}%  '
          f'(drop: {row["soh_drop"]:.1f}%)')

## 5. SOH Degradation Plot

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for bid in sorted(feat['battery_id'].unique()):
    grp = feat[feat['battery_id'] == bid].sort_values('cycle')
    color = 'red' if bid in non_mono else '#1976D2'
    alpha = 0.5 if bid in non_mono else 0.7
    ax.plot(grp['cycle'], grp['soh'], color=color, alpha=alpha, linewidth=1)

ax.axhline(EOL_THRESHOLD, color='black', linestyle='--', linewidth=2,
           label=f'EOL threshold ({EOL_THRESHOLD}%)')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#1976D2', label='Monotonic degradation'),
    Line2D([0], [0], color='red',     label='Non-monotonic (mixed protocol)'),
    Line2D([0], [0], color='black', linestyle='--', label='EOL threshold (70%)'),
]
ax.legend(handles=legend_elements, loc='lower left')
ax.set_xlabel('Discharge Cycle')
ax.set_ylabel('SOH (%)')
ax.set_title('SOH Degradation — All 34 Batteries')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'plots', 'soh', 'nb05_soh_degradation.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 6. SOH Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(feat['soh'], bins=40, color='#1976D2', edgecolor='white', linewidth=0.5)
axes[0].axvline(EOL_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
                label='EOL (70%)')
axes[0].set_xlabel('SOH (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of SOH Across All Cycles')
axes[0].legend()

axes[1].boxplot(
    [feat[feat['battery_id'] == b]['soh'].values
     for b in sorted(feat['battery_id'].unique())],
    labels=sorted(feat['battery_id'].unique()),
    patch_artist=True, medianprops=dict(color='red'),
    boxprops=dict(facecolor='lightblue')
)
axes[1].axhline(EOL_THRESHOLD, color='red', linestyle='--', linewidth=1)
axes[1].set_xticklabels(sorted(feat['battery_id'].unique()),
                         rotation=90, fontsize=7)
axes[1].set_ylabel('SOH (%)')
axes[1].set_title('SOH Distribution per Battery')

plt.tight_layout()
plt.show()

## 7. Correlation: SOH vs Features

In [ ]:
feat_cols = ['discharge_duration', 'energy_Wh', 'voltage_mean', 'voltage_min',
             'voltage_std', 'current_mean', 'temperature_mean', 'temperature_max',
             'ambient_temp_C', 'cycle']

corrs = feat[feat_cols + ['soh']].corr()['soh'].drop('soh').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#C62828' if v < 0 else '#1565C0' for v in corrs.values]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with SOH')
ax.set_title('Feature–SOH Correlation')
plt.tight_layout()
plt.show()

print('Feature correlations with SOH:')
print(corrs.round(3).to_string())

## 8. Save Dataset with SOH Column

In [ ]:
final_partial = feat[['battery_id', 'cycle', 'capacity', 'initial_capacity', 'soh',
                       'discharge_duration', 'energy_Wh',
                       'voltage_mean', 'voltage_min', 'voltage_std',
                       'current_mean', 'temperature_mean', 'temperature_max',
                       'ambient_temp_C']].copy()

out = os.path.join(ROOT, 'cleaned_data', 'final_dataset_soh_only.csv')
final_partial.to_csv(out, index=False)
print(f'Saved: {out}')
print(f'Shape: {final_partial.shape}')
print('\nSOH target summary:')
print(final_partial['soh'].describe().round(3).to_string())